# Proyecto 1 — Analítica de Datos
## Modelamiento de la precipitación semanal en el Valle del Cauca

**Universidad Autónoma de Occidente** · Ingeniería de Datos e Inteligencia Artificial  
Analítica de Datos 2026-2S · Prof. Johann A. Ospina  
César Armando Reyes Oliveros — 2236379 · Yesenia Díaz — 2231783 · Juan Pablo Maya

---

### Para qué sirve este cuaderno

Registra **qué** hacemos, **por qué**, en qué **material de clase** se apoya cada
decisión y qué **resultado** dio. Sirve para que el grupo siga el hilo y para
preparar la sustentación.

> ⚠️ **No es el entregable.** El enunciado exige **PDF de máximo 10 páginas** más
> **un único `.R`**, y prohíbe R Markdown o solo-Colab. El código canónico vive en
> **`R/proyecto1.R`**; aquí solo se explica.


---
## Cómo trabajar en RStudio

🔴 **Empieza SIEMPRE cada sesión con esto:**


In [ ]:
source("R/proyecto1.R")


Corre el análisis completo (~30 s) y deja en memoria todo lo que necesitas:

| Objeto | Qué es |
|---|---|
| `datos` | las 70 estaciones, 9 columnas |
| `borde` | polígono del departamento |
| `precip`, `r_alt`, `mascara` | rasters recortados al área |
| `modelo_tendencia` | el `lm()` ajustado |
| `figura()`, `mapa_base()`, `esquina_leyenda()` | helpers de dibujo |
| `col_banda`, `pal_lon`, `cortes_x`, `lat0`, `lon0` | auxiliares de las figuras |

> ⚠️ **Este fue el error del 11-sep.** Se intentó correr los comandos de las figuras
> sin haber ejecutado antes el modelo de tendencia. `datos$residual` no existía y R
> respondió `non-numeric argument to mathematical function` — un mensaje que no dice
> nada sobre la causa real. Con `source()` primero, no vuelve a pasar.


---
## Paso 0 — El encargo, el molde y los datos

### Qué pide el enunciado

Modelar la precipitación semanal en el Valle del Cauca en función de covariables
ambientales y **evaluar el papel de la correlación espacial**. Cuatro bloques:
EDA → Modelación → Validación → Predicción, justificando cada uno paso a paso.

La restricción que manda sobre todo:

> *"Utilizar únicamente los scripts, procedimientos y metodologías trabajados en
> clase. Librerías adicionales permitidas solo para: elaboración de gráficos y carga
> de mapas o archivos de Excel."*

### El molde

**`Ejemplo3_Geoestadística.R`** es el más cercano: CHIRPS semanal, Valle del Cauca,
borde GADM, kriging. Es el mismo problema. **`Ejemplo4_Geoestadística.R`** tiene el
flujo completo mejor ordenado. Ambos en `material_clase/Practice Geostatistics/`.

El profesor dejó una nota en la cabecera del Ejemplo3 que conviene tener presente:

> *"las funciones están con la teoría vista en clase son correctas, sin embargo, no
> quiere decir que los resultados sean apropiados. **Usted debe leer y ajustar lo
> necesario** para que los resultados sean apropiados"*

| Sección del molde | Qué hace | Nuestro paso |
|---|---|---|
| Cargar imágenes | `rast()` | 1 |
| Borde del valle | `as.polygons()` sobre la máscara | 2.1 |
| Puntos de muestreo | `spatSample(method = "random")` | 2.2 |
| Proyección | grados → km, coseno de la latitud | 2.2 |
| EDA espacial | histogramas, posting, dispersión | 3 |
| Extraer tendencia | `lm(z ~ x_km + y_km)` | 4 |
| Distancias | `dist()` | 5 |
| Semivariograma | nube + bins con `tapply()` | 6 |
| Ajuste de modelo | `optim()` L-BFGS-B multi-arranque | 6 |
| Predicción | sistema kriging con `solve()` | 7 |
| Validación cruzada | LOOCV a mano | 8 |

### Librerías

✅ Con precedente: `terra`, `sp`, `gstat`, `geodata`, `chirps`, `ggplot2`, `gridExtra`,
`geostan`, `openxlsx`, `dplyr`, y base R (`lm`, `optim`, `solve`, `dist`, `tapply`, `cut`).

❌ Sin precedente: `automap`, `spdep`, `sf`, `caret`, `randomForest`, `mgcv`, `tmap`,
`leaflet`, el paquete viejo `raster`, y `terra::interpIDW`.

**Decisión: solo `terra` + base R.** Es el subconjunto más defendible y es lo que usa
el molde.

### Verificación de los datos

```
md5  d1bd258c058004a7877617817027b60a   datos_proyecto_1.zip   respaldo == repo ✅
md5  813557728ea4ceee9b5977e37bc68789   enunciado PDF          idéntico      ✅
CRC  229 de 229 .tif extraídos coinciden con el zip                          ✅
```


---
# Paso 1 — Auditoría de los datos

**Qué.** Verificar geometría, valores centinela y cobertura real antes de modelar.

**Por qué.** El enunciado afirma que las capas son *"directamente comparables píxel
a píxel"*. Cierto en geometría, **falso en cobertura**.

**Material.** `Ejemplo4` (carga con `rast`), `carpeta/Script_Class5.R` (`terra`).


## 1.1 Geometría — se cumple

```
              filas cols bandas  res   xmin  xmax  ymin  ymax
Precipitacion    39   37     52 0.05 -77.55 -75.7  3.05  5.00
Temperatura      39   37     52 0.05 -77.55 -75.7  3.05  5.00
Radiacion        39   37     52 0.05 -77.55 -75.7  3.05  5.00
Altitud          39   37      1 0.05 -77.55 -75.7  3.05  5.00
```

39 × 37 = **1 443 píxeles** de 0,05° (≈ 5,6 km), EPSG:4326. Las 52 bandas son las
semanas ISO.


## 1.2 Valores centinela

CHIRPS codifica "sin dato" como un negativo enorme, no como `NA`. La lluvia no puede
ser negativa; si no se filtra, **todo lo que sigue es basura**.

```
Mínimo crudo:     -76867.3
Celdas negativas:      104
Mínimo tras limpiar:  2.77 mm   ← valor físico
```


In [ ]:
r_precip[r_precip < 0] <- NA


## 1.3 La máscara del área de estudio

El rectángulo de 1 443 píxeles **no** es el departamento: incluye océano Pacífico y
territorio vecino. Sin máscara no hay denominador honesto.

> 🔴 **El error que cometimos.** La primera versión midió cobertura contra las 1 443
> celdas del rectángulo y dio `Altitud → 47,7 % → SE DESCARTA`, descartando la mejor
> covariable.
> 
> ```
> altitud / rectángulo   = 688/1443 = 47,7 %  →  "media cobertura"
> altitud / departamento =  688/688 =  100 %  →  "cobertura total"
> ```
> 
> **Define la máscara antes de calcular cualquier porcentaje.**


In [ ]:
mascara <- !is.na(r_alt) & !is.na(r_precip[[SEMANA]])
N_VALLE <- sum(values(mascara), na.rm = TRUE)   # 688 celdas


## 1.4 Cobertura efectiva — el hallazgo que define el modelo

| Variable | Celdas | % del área | Valores distintos | |
|---|---|---|---|---|
| Precipitación | 688 | 100,0 % | 688 | ✅ |
| Altitud | 688 | 100,0 % | 688 | ✅ |
| Temperatura | 123 | 17,9 % | 56 | ❌ |
| **Radiación** | **28** | **4,1 %** | **3** | ❌ |

**Por qué.** CHIRPS es nativo 0,05°; NASA POWER es nativo **0,5°**, diez veces más
grueso. Al llevarlo a la grilla fina solo sobrevivieron los centros originales. El
enunciado dice "remuestreadas a la resolución de CHIRPS", pero eso fue reproyección
de rejilla, **no relleno**.

**Por qué importa.** La radiación, dentro del Valle, son **3 números**. No es un
gradiente: es un escalón costa/valle/cordillera. Meterla al `lm()` como continua le
atribuye a "radiación" lo que es **posición geográfica**, que ya entra como `x_km`.

**Decisión.** Temperatura y radiación **fuera**. El modelo usa **altitud +
coordenadas**, las tres con 100 % de cobertura y cero dato inventado.

Esto además evita `terra::interpIDW`, que es como se rellenarían esos huecos y que
**no aparece en ningún script de clase** (verificado con `grep` sobre `material_clase/`).


## 1.5 Elección de la semana — con evidencia

"Octubre es lluvioso" es razonable, pero el dato decide. Se promedian las 52 semanas
y se toma el máximo.

```
Semana más lluviosa (climatología 2010-2025): 44 (91,9 mm)
Semana más seca:                               3 (28,7 mm)
```

La **44** (principios de noviembre), no la 42.


In [ ]:
media_semanal <- sapply(1:nlyr(r_precip),
                        function(k) mean(values(r_precip[[k]]), na.rm = TRUE))
SEMANA <- which.max(media_semanal)   # -> 44


![Ciclo anual](resultados/fig01_ciclo_anual.png)

**Interpretación.** Régimen **bimodal** andino: dos picos al año (abril-mayo y
octubre-noviembre) por el doble paso de la Zona de Convergencia Intertropical. El
mínimo de la semana 3 es el veranillo de enero. La semana elegida tiene más del
triple de lluvia que la más seca, así que la señal espacial será fuerte.


---
# Paso 2 — Puntos de muestreo

## 2.1 El borde real del departamento

**Qué.** Convertir la máscara en polígono. **Por qué.** Los puntos hay que sortearlos
*dentro* del departamento; sobre el rectángulo caerían en el Pacífico.

**Material.** `Ejemplo4_Geoestadística.R`, líneas 21-24.


In [ ]:
borde <- as.polygons(mascara, dissolve = TRUE)
borde <- borde[borde[[1]] == 1, ]      # <- la línea que la gente olvida

# CHIRPS (787 celdas) desborda el departamento (688) y cubre mar abierto.
precip <- mask(r_precip[[SEMANA]], mascara, maskvalues = c(FALSE, NA))


### La línea que la gente olvida

`as.polygons()` sobre un raster lógico devuelve **dos** polígonos: el de los `TRUE`
(atributo 1) y el de los `FALSE` (atributo 0). Sin `borde[borde[[1]] == 1, ]` el
"borde" incluye el océano.

### Verificación

```
Polígonos devueltos por as.polygons(): 2
Área del borde conservado:        21 125 km²   (real: 22 140 km² → 95 %)
Celdas de CHIRPS antes del recorte:  787
Celdas tras recortar al área:        688
```


![Borde del Valle](resultados/fig02_borde_valle.png)

**Interpretación — el hallazgo central del EDA.**

| Zona | Lluvia semana 44 |
|---|---|
| Vertiente pacífica (Buenaventura, verde) | **200+ mm** |
| Valle interandino (blanco) | **30-50 mm** |

Factor de **5 a 7×** en ~150 km. Efecto orográfico: la humedad del Pacífico choca
contra la cordillera Occidental, descarga en la vertiente y llega seca al valle.

**Consecuencia:** buena parte de la varianza es **tendencia de gran escala en la
longitud**, no estructura aleatoria. Por eso el modelo se parte en dos: una tendencia
determinística y un residuo espacialmente correlacionado que interpola el kriging.


## 2.2 Sorteo de las estaciones

**Qué.** Sortear 70 celdas dentro del borde y proyectar a kilómetros.

**Por qué no usar las 688 celdas:**

1. **Costo.** El sistema kriging invierte una matriz (n+1)×(n+1) por cada punto
   predicho y por cada iteración del LOOCV. Con n = 688 es inviable.
2. **Sentido.** Un semivariograma modela un proceso muestreado en **estaciones**. Con
   las 688 celdas ya tienes el mapa y no hay nada que interpolar.

**Material.** `Ejemplo4` sortea 80 puntos; `Ejemplo3` sortea 40. El muestreo aleatorio
de la grilla **es el idioma del profesor**, no un atajo.

> ⚠️ **Esto hay que escribirlo en el informe**, porque es la crítica obvia en la
> sustentación: las 688 celdas vienen de un producto grillado (CHIRPS), no de
> estaciones reales. El muestreo **simula** una red de observación para poder evaluar
> el kriging contra un valor conocido. El R² de validación mide **capacidad de
> reconstrucción**, no destreza predictiva frente a datos independientes.


In [ ]:
set.seed(2026)
puntos <- spatSample(borde, size = 70, method = "random")

datos <- data.frame(
  lon     = crds(puntos)[, 1],
  lat     = crds(puntos)[, 2],
  precip  = extract(precip, puntos)[, 2],   # [,2] porque extract() antepone ID
  altitud = extract(r_alt,  puntos)[, 2]
)
datos <- na.omit(datos)

# Proyección a km. El variograma mide semivarianza contra DISTANCIA:
#   1 grado de latitud  = 110.574 km (constante)
#   1 grado de longitud = 111.320 km * cos(latitud)
lat0 <- mean(datos$lat); lon0 <- mean(datos$lon)
datos$x_km <- (datos$lon - lon0) * 111.320 * cos(lat0 * pi / 180)
datos$y_km <- (datos$lat - lat0) * 110.574


**Resultado:** 70 estaciones, ningún `NA`, **2 415 pares** para el semivariograma
(el molde trabaja con 780).


---
# Paso 3 — Análisis exploratorio espacial

## 3.1 Distribución de la precipitación

```
Min  34.93   Q1  51.72   Mediana  64.17   Media  91.09   Q3 125.88   Max 189.01
Desviación estándar : 51.32 mm
Coeficiente de variación : 56.3 %
Media / mediana : 1.42
Shapiro-Wilk : W = 0.8156, p = 6.4e-08  ->  SE RECHAZA normalidad
```

**Interpretación.** La media es 42 % mayor que la mediana: asimetría fuerte a la
derecha. De Q1 a la mediana hay 12 mm; de la mediana a Q3 hay 62 mm — cinco veces
más. La cola derecha es la vertiente pacífica.

Esto importa porque el kriging es un predictor **lineal**, óptimo bajo normalidad.
Pero la pregunta correcta no es si `precip` es normal, sino si lo son **los
residuales** después de quitar la tendencia. Se resuelve en el Paso 4.


![Distribución](resultados/fig03_distribucion.png)

## 3.2 Mapa de posting

![Posting](resultados/fig04_posting.png)

Los símbolos grandes se concentran al oeste. Confirma visualmente el gradiente.


## 3.3 Relación con las covariables

```
Correlación de Pearson con precip:
  x_km    -0.890      ← la longitud sola explica 0.890² = 79 % de la varianza
  altitud -0.651      ← sospechosa, ver Paso 4
  y_km    -0.274
```


![Covariables](resultados/fig05_covariables.png)

**Interpretación.** La longitud domina. La altitud parece fuerte pero está confundida
con ella (la costa es baja *y* occidental). La latitud aporta poco por sí sola.


---
# Paso 4 — Modelo de tendencia

Kriging universal = **tendencia determinística + residuo espacialmente correlacionado**.
Aquí se estima la primera parte.

## 4.1 La confusión entre altitud y longitud

```
cor(altitud, precip) = -0.651    (simple)
cor(altitud, x_km)   =  0.755    (colinealidad)

Modelo A:  precip ~ x_km + y_km            R² = 0.8336   R²adj = 0.8286
Modelo B:  precip ~ x_km + y_km + altitud  R² = 0.8449   R²adj = 0.8379

  (Intercept)  79.4979   p = 0.0000
  x_km         -1.1610   p = 0.0000
  y_km         +0.3165   p = 0.0000   ← correlación simple era -0.274
  altitud      +0.0098   p = 0.0317   ← ¡signo invertido!

Test F anidado A vs B:  F = 4.8166,  p = 0.0317  →  la altitud se queda
```

**Qué significa.** La correlación simple decía *"más alto = más seco"*. Lo que decía
en realidad era *"más al este = más seco"*: la costa es baja **y** occidental, la
cordillera es alta **y** oriental. Al controlar la posición, el signo se invierte.

Es un caso de libro de **confusión (confounding)**.


### Visto en cuatro figuras

![Colinealidad](resultados/fig07a_colinealidad.png)

**(A) La causa.** `r = 0,755`. No hay puntos verdes altos ni rojos bajos — **no existe
costa alta ni valle a nivel del mar**. Por eso el modelo no puede separarlas del todo.


![Correlación simple](resultados/fig07b_correlacion_simple.png)

**(B) El engaño.** La recta baja, pero los colores delatan el truco: el extremo
izquierdo (altitud ≈ 0, lluvia 180 mm) es **todo verde**. La pendiente negativa la
produce un grupo que es bajo *y* occidental.


![Estratificado](resultados/fig07c_estratificado.png)

**(C) La realidad, más rica de lo esperado.**

| Banda | Pendiente | Física |
|---|---|---|
| 🟢 Oeste (n=24) | **−0,0582** | Chocó: el máximo de lluvia está **al nivel del mar**; la masa húmeda es somera y descarga abajo. Subiendo, seca |
| 🟠 Centro (n=23) | **+0,0167** | Ladera de la cordillera Occidental: ascenso orográfico clásico |
| 🔴 Este (n=23) | **+0,0050** | Valle del Cauca y ladera de la Central: mismo mecanismo, más débil |
| Agrupado (n=70) | −0,0369 | promedio engañoso |

> ⚠️ **Limitación que hay que declarar.** El efecto de la altitud **no es homogéneo en
> el espacio**: son dos regímenes físicos opuestos. Lo correcto sería una interacción
> `altitud × longitud`, pero **ningún script de clase usa interacciones** — los cuatro
> ejemplos son `lm(z ~ x + y)` pelado. Se mantiene la deriva lineal y la limitación
> queda documentada.


![Variable añadida](resultados/fig07d_variable_anyadida.png)

**(D) El efecto parcial.** Panel canónico para confusión. Ejes = residuales de
`precip` y de `altitud` después de quitarle a ambos la posición. La pendiente,
**+0,0098**, es *exactamente* el coeficiente del `lm()` múltiple.


## 4.2 Diagnóstico de residuales — valida todo el enfoque

```
precip cruda  ->  Shapiro W = 0.8156,  p = 6.4e-08   ->  NO normal
residuales    ->  Shapiro W = 0.9750,  p = 0.1739    ->  normal ✅
sd:  51.32 mm  ->  20.21 mm
```

**Contesta la pregunta del Paso 3: no hay que transformar la variable.** La asimetría
de `precip` era la tendencia, no la distribución. Quitada la tendencia, los residuales
son normales y el kriging queda justificado como predictor lineal óptimo.


![Residuales](resultados/fig06_residuales.png)

**Panel (B)** el Q-Q sigue la recta con colas ligeramente cortas — aceptable.

⚠️ **Panel (C) muestra un problema real.** Los residuales dibujan una **U**:

```
ajustado ~30   → residual +30
ajustado ~100  → residual -40
ajustado ~170  → residual +25
```

Curvatura sistemática: la tendencia lineal en `x_km` está mal especificada, la
relación real es curva.

**Decisión.** Ningún script de clase usa tendencia polinómica, así que agregar
`I(x_km^2)` sacaría del whitelist. Se mantiene lineal — y la U es *estructura espacial
que la tendencia determinística no capturó*, que es **exactamente lo que el kriging
debe absorber**. El enunciado pide "evaluar el papel de la correlación espacial":
esa U **es** la evidencia de que importa.


## 4.3 Las mismas relaciones, sobre el mapa

![Mapa bandas](resultados/fig08a_mapa_bandas.png)

Las tres bandas de longitud usadas en el análisis estratificado, sobre el territorio.


![Mapa altitud](resultados/fig08b_mapa_altitud.png)

Relieve con las 70 estaciones encima. Se ven las dos cordilleras y el valle entre ellas.


![Mapa residuales](resultados/fig08c_mapa_residuales.png)

**🔴 La figura más importante del EDA.** Los residuales están **agrupados en manchas**,
no dispersos al azar:

| Zona | Color | Lectura |
|---|---|---|
| Centro del valle (−76,7 · 3,5-3,8) | 🔵 grandes | el modelo **sobreestima** |
| Flanco oriental (−75,9 · 3,9-4,1) | 🔴 grandes | el modelo **subestima** |
| Ladera pacífica (−77,2 · 3,7-4,0) | 🔴 | subestima |
| Norte (−76,2 · 4,5-4,9) | 🔵 | sobreestima |

Es la **U del panel (C) vista en el espacio**. La tendencia lineal no puede curvarse,
así que aplana el fondo del valle hacia arriba y las laderas hacia abajo.

Anticipa el resultado del Paso 5: **el Moran sobre residuales será claramente positivo.**


![Obs vs tendencia](resultados/fig08d_mapa_obs_vs_tendencia.png)

Observado contra ajustado. El modelo reproduce el gradiente general pero suaviza los
extremos — visible en que los símbolos del panel (B) son más uniformes.


---
# Paso 5 — Autocorrelación espacial

**Es el corazón del enunciado:** *"evaluar el papel de la correlación espacial"*.

**Material.** `carpeta/Script_Spatial_Analytics.R` calcula Moran y Geary a mano.
La `Lecture_SpatialStatistics.pdf` da las fórmulas y, sobre todo, esto:

> *"OLS assumes independent error terms. When residuals exhibit spatial
> autocorrelation, **the standard OLS assumption is violated**, and estimates
> become inefficient or biased"*
> 
> *"**Diagnostic: Moran's I applied to OLS residuals is the standard test** for
> spatial autocorrelation in a regression context"*

O sea: lo que hacemos aquí es literalmente el procedimiento que prescribe la clase.


## 5.1 🔴 Un bug en el script del profesor

`Script_Spatial_Analytics.R`, línea 77, estandariza la matriz de pesos así:

```r
W <- matrix(ifelse(row_sums == 0, 0, W / row_sums), nrow = n, ncol = n)
```

`ifelse()` devuelve un objeto del largo del **test**, no del resultado. `row_sums`
mide `n`, así que devuelve `n` valores en vez de `n²`: se queda con la primera
columna de `W/row_sums` y `matrix()` la recicla en todas las demás. **La
estandarización por filas no ocurre.**

```
método del script:  suma por fila = 0 2 0 0   (debería ser 1 1 0 0)
con sweep():        suma por fila = 1 1 0 0   ✅
```

No invalida el curso — la teoría del script está bien y es un *gotcha* clásico de R.
Pero copiado tal cual, el Moran sale mal. **Usamos `sweep()`.**


In [ ]:
pesos_espaciales <- function(D, q) {
  n <- nrow(D)
  umbral <- quantile(D[upper.tri(D)], q)
  W <- matrix(0, n, n); W[D <= umbral] <- 1; diag(W) <- 0
  filas <- rowSums(W)
  list(W = sweep(W, 1, ifelse(filas == 0, 1, filas), "/"),   # <- sweep, no ifelse
       umbral = unname(umbral), vecinos = mean(filas))
}

moran <- function(z, W) {
  n <- length(z); zc <- z - mean(z)
  (n / sum(W)) * (sum(W * outer(zc, zc)) / sum(zc^2))
}

geary <- function(z, W) {
  n <- length(z); zc <- z - mean(z)
  ((n - 1) / (2 * sum(W))) * (sum(W * outer(z, z, function(a,b) (a-b)^2)) / sum(zc^2))
}


## 5.2 El resultado que contesta el enunciado

El umbral de vecindad cambia el resultado, así que se reporta un **rango**, no un
número suelto. (El profesor usa el percentil 5; el script de los compañeros, el 25.)
Significancia por permutación con 999 remuestreos.

```
                     I_precip  C_precip  I_residual  C_residual    p
percentil  5 (20 km)   0.9552   0.0351     0.7931      0.1871   0.001
percentil 10 (28 km)   0.9376   0.0610     0.7217      0.2746   0.001
percentil 25 (48 km)   0.8125   0.1574     0.4598      0.5398   0.001

Esperado bajo aleatoriedad:  E[I] = -0.0145      E[C] = 1
```

**Sí importa la correlación espacial, y muchísimo.** Después de que el `lm()` se
llevó el 84 % de la varianza, lo que queda **sigue teniendo estructura espacial
fuerte**. Moran y Geary apuntan al mismo lado (I alto ↔ C bajo), así que no hay
error de cálculo.

`p = 0,001` es el mínimo posible con 999 permutaciones: en 999 reetiquetados al
azar **ninguno** alcanzó el I observado.

| Índice | Sin autocorrelación | Autocorrelación positiva |
|---|---|---|
| Moran I | ≈ −1/(n−1) = −0,014 | → +1 |
| Geary C | = 1 | → 0 |


![Moran scatter](resultados/fig09_moran_scatter.png)

**Diagrama de Moran.** Eje x: residual estandarizado. Eje y: promedio de sus
vecinos. **La pendiente de la recta ES el índice de Moran.** Rojo = alto rodeado de
alto, azul = bajo rodeado de bajo, gris = discordante. El predominio de rojo y azul
sobre gris es la autocorrelación positiva vista punto a punto.


## 5.3 El correlograma — lo más informativo del paso

![Correlograma](resultados/fig10_correlograma.png)

```
h ≈  10 km   I = +0.79    dependencia local fuerte
h ≈  30 km   I = +0.50
h ≈  48 km   I = -0.05    ← cruza cero
h ≈  89 km   I = -0.54    ← anticorrelación máxima
h ≈ 120 km   I =  0.00    ← vuelve a cruzar
h ≈ 149 km   I = +0.51
```

**No es una curva que decae y se queda en cero: es una onda.** Y cada tramo se lee
directamente sobre el mapa de residuales:

| Tramo | Qué significa |
|---|---|
| 0-48 km, positivo | Dependencia espacial genuina. **Es el rango que estimará el semivariograma** |
| ~89 km, I = −0,54 | Puntos separados ~89 km tienen residuales de **signo opuesto**: media anchura del departamento, el azul del valle contra el rojo de un flanco |
| ~149 km, positivo | Los **dos** flancos son rojos, y están separados ~150 km |

⚠️ **Consecuencia.** El lóbulo negativo dice que los residuales **no son
estacionarios de segundo orden**: queda estructura determinística. El semivariograma
exponencial asume decaimiento monótono hasta una meseta, y eso solo se cumple
**hasta ~50 km**. Ahí se fija el cutoff del Paso 6 — con respaldo del libro.


---
# Paso 6 — Semivariograma

## 6.1 🔴 Corrección: el libro sí respalda la tendencia polinómica

En el Paso 4 se dijo que la tendencia cuadrática estaba fuera del whitelist porque
ningún *script* la usa. Cierto sobre los scripts. **Falso sobre el libro**, que dice
textualmente:

> *"En caso de encontrarse tendencia en los datos, esta tendencia se modela con
> **modelos de regresión polinómicos** o con análisis a dos vías y el semivariograma
> se construye con los residuales obtenidos"*

Y sobre la distancia máxima da dos reglas:

> *"en general, se usan los rezagos espaciales **hasta la mitad de la máxima
> distancia**"*

> *"Es importante elegir una distancia máxima [...] **en caso de que se observe un
> comportamiento errático a distancias mayores**"*

⚠️ El script de los compañeros usa `cutoff <- max(D) * 0.75`. El libro dice **mitad**.

**Decisión:** se llevan **las dos tendencias en paralelo** y decide la evidencia.


## 6.2 Tres modelos, no uno

La `Lecture` define **tres** modelos válidos (ec. 12-14), parametrizados por pepita
`c0`, meseta `c0+c1` y rango `a`. Los scripts solo ajustan el exponencial.
**Ajustamos los tres.**

| Modelo | γ(h) |
|---|---|
| Esférico | `c0 + c1[1.5(h/a) − 0.5(h/a)³]` para h ≤ a; `c0+c1` si h > a |
| Exponencial | `c0 + c1[1 − exp(−h/a)]` |
| Gaussiano | `c0 + c1[1 − exp(−h²/a²)]` |

El ajuste es por mínimos cuadrados sobre el semivariograma empírico (libro §3.1),
con `optim()` L-BFGS-B y varios arranques porque los modelos no son lineales en los
parámetros — los valores iniciales salen de *"estimación a ojo o a sentimiento"*
sobre el variograma empírico, como dice el libro.


In [ ]:
mod_esferico <- function(h, c0, c1, a)
  ifelse(h <= a, c0 + c1 * (1.5*(h/a) - 0.5*(h/a)^3), c0 + c1)
mod_exponencial <- function(h, c0, c1, a) c0 + c1 * (1 - exp(-h/a))
mod_gaussiano   <- function(h, c0, c1, a) c0 + c1 * (1 - exp(-(h/a)^2))


## 6.3 El diagnóstico que zanja la discusión

Bajo **estacionariedad de segundo orden** (libro, Definición 4) la meseta debe
aproximar la varianza del proceso. Si la supera, el semivariograma sigue creciendo:
queda tendencia sin modelar.

```
                        meseta  varianza  razón   veredicto
cuadrática  50 km  gauss  152.9   158.8    0.96   coherente ✅
cuadrática  50 km  esfér  161.7   158.8    1.02   coherente ✅
cuadrática 113 km  esfér  176.2   158.8    1.11   coherente ✅
lineal      50 km  gauss  537.6   408.4    1.32   NO estacionario ❌
lineal      50 km  esfér  769.6   408.4    1.88   NO estacionario ❌
lineal      50 km  expon 1275.1   408.4    3.12   NO estacionario ❌
```

**Lineal: falla en las 6 configuraciones. Cuadrática: coherente en 5 de 6.**

Ya no es "el libro dice" contra "los scripts dicen". Es **evidencia propia**: la
tendencia lineal viola el supuesto que el propio material exige.


![Variograma lineal](resultados/fig11a_variograma_lineal.png)

**Tendencia lineal.** El semivariograma **sube sin parar** hasta los 50 km y cruza la
varianza muestral (línea punteada). No hay meseta. Firma de libro de tendencia sin
modelar.


![Variograma cuadrática](resultados/fig11b_variograma_cuadratica.png)

**Tendencia cuadrática.** **Sí forma meseta**, y se estabiliza justo en la línea de
la varianza (159) hacia los 40 km. Los tres modelos convergen. Esto es un
semivariograma sano.


![Variograma cutoff libro](resultados/fig11c_variograma_cuadratica_libro.png)

**Sensibilidad al cutoff.** La misma tendencia cuadrática ajustada hasta 113 km (la
regla general del libro). La meseta sube de 161,7 a 176,2 — razón 1,11 en vez de
1,02. Sigue siendo coherente, pero el ajuste a 50 km es mejor, y el correlograma
justifica por qué.


---
# Paso 7 — Kriging

**Material.** `Lecture_SpatialStatistics.pdf`:

> *"Kriging is a geostatistical interpolation method that provides the **Best Linear
> Unbiased Predictor (BLUP)** of Z at an unsampled location s₀"*

ec. (15) `Ẑ(s₀) = Σ λᵢ Z(sᵢ)` · ec. (16) los pesos minimizan la varianza de
predicción sujeto a `Σ λᵢ = 1`, resuelto con multiplicador de Lagrange.

Se predice el **residuo** por kriging ordinario y se le suma la **tendencia**
(libro, Def. 5: `Y(s) = μ(s) + Z(s)`).

## 7.1 Una corrección a la covarianza de los scripts

Los scripts de clase usan `C(h) = meseta · exp(−h/φ)`. Eso solo es correcto para el
**exponencial sin pepita**. La forma general es:

```r
C(h) = meseta − γ(h)
```

Así el efecto pepita entra bien: `C(0) = meseta`, y `C(0⁺) = meseta − c0 = c1`.
Y sirve con cualquiera de los tres modelos.


In [ ]:
kriging_residuo <- function(s0, coords_obs, resid_obs, Sigma, C_fn) {
  n_obs <- nrow(coords_obs)
  d0 <- sqrt((coords_obs[,1] - s0[1])^2 + (coords_obs[,2] - s0[2])^2)
  c0_vec <- C_fn(d0)
  A <- rbind(cbind(Sigma, rep(1, n_obs)), c(rep(1, n_obs), 0))   # Lagrange
  sol <- solve(A, c(c0_vec, 1))
  lambda <- sol[1:n_obs]; mu <- sol[n_obs + 1]
  c(pred = sum(lambda * resid_obs),
    var  = max(0, C_fn(0) - sum(lambda * c0_vec) - mu))
}


## 7.2 🔴 El gaussiano es una trampa

Primer intento eligiendo el modelo por **suma de cuadrados** (el que mejor ajusta el
semivariograma). Resultado:

```
lineal      RMSE 21.32 → 160.29    R²_CV = -8.90
cuadrática  RMSE 14.02 →  23.16    R²_CV =  0.79
sd_z esperado ≈ 1, obtenido 339 y 18.6
```

**El kriging empeoraba la predicción.** Causa:

```
                          κ(A)      autovalor mínimo de Σ
lineal      esférico    1.13e+06      +16.9
lineal      gaussiano   1.19e+08      +0.0000476   ← casi singular
cuadrática  esférico    1.01e+04       +9.13
cuadrática  gaussiano   1.23e+05       +0.00856    ← idem
```

El gaussiano deja la matriz de kriging casi singular y los pesos se disparan.

**Lección: ajustar mejor el variograma ≠ predecir mejor.** El criterio de selección
pasa a ser el desempeño en validación cruzada.


## 7.3 El modelo de menor RMSE **no** es el que se usa

```
tendencia   modelo       RMSE   R²_CV    sd_z    κ(A)
lineal      exponencial  7.09   0.9806   0.69    7.7e6   ← menor RMSE
lineal      esferico     7.15   0.9803   0.73    1.1e6
cuadratica  exponencial  8.72   0.9707   1.01    1.7e5
cuadratica  esferico     8.86   0.9697   1.06    1.0e4
cuadratica  gaussiano   23.16   0.7933  18.61    1.2e5
lineal      gaussiano  160.29  -8.8974 339.00    1.2e8
```

`sd_z` es la desviación del error estandarizado `(obs − pred)/√(var kriging)`. **Si
la varianza de kriging está bien calibrada, debe valer ~1.**

Tres filtros, los tres necesarios:

| Filtro | Por qué |
|---|---|
| **estacionariedad** | meseta ≈ varianza (libro, Def. 4) |
| **calibración** | `sd_z` ∈ [0,8 · 1,25], o el mapa de incertidumbre no significa nada |
| **estabilidad** | κ(A) acotado |

```
              razón  estacionario calibrado estable  RMSE   válida
cuadratica esferico  1.02     TRUE      TRUE    TRUE   8.86   ✅
lineal  exponencial  3.12    FALSE     FALSE   FALSE   7.09   ❌
cuadratica exponenc  1.80    FALSE      TRUE    TRUE   8.72   ❌
lineal    gaussiano  1.32    FALSE     FALSE   FALSE 160.29   ❌
```

**Exactamente una combinación pasa las tres.**

⚠️ `lineal + exponencial` baja el RMSE a 7,09 mm, pero su meseta **triplica** la
varianza y su `sd_z = 0,69` haría que el mapa de incertidumbre mintiera. Se descarta
con argumento, no por gusto.

```
SELECCIÓN FINAL: tendencia cuadrática + semivariograma esférico
  pepita 0.0 | meseta 161.7 mm² | rango 56.1 km
  RMSE 8.86 mm | MAE 5.53 mm | R²_CV 0.9697 | sd_z 1.06 | κ 1.0e4
```


![Validación cruzada](resultados/fig12_validacion_cruzada.png)

**(A)** Observado contra predicho, sobre la diagonal. **(B)** Residuos sin patrón.
**(C)** Q-Q del error estandarizado: sigue la recta y su desviación es 1,06 — la
varianza de kriging está bien calibrada, así que el mapa de incertidumbre es
interpretable.


![Mapa error LOOCV](resultados/fig13_mapa_error_loocv.png)

**Dónde falla el modelo.** A diferencia del mapa de residuales de la tendencia
(Paso 4), aquí los colores **ya no forman manchas**: el kriging absorbió la
estructura espacial. Eso es la confirmación visual de que hizo su trabajo.


---
# Paso 8 — Predicción

Kriging sobre las **688 celdas** del departamento. Tres niveles de evidencia:

```
LOOCV sobre las 70 estaciones      RMSE  8.86 mm   R² 0.970
620 celdas SIN estación            RMSE 10.79 mm   R² 0.950   ← el número honesto
68 celdas CON estación             RMSE  2.09 mm
```

El 2,09 mm no es mérito: con pepita 0 el kriging **interpola exacto** en los datos.
El número que vale es el de las celdas no muestreadas.

⚠️ **Detalle que hay que declarar:** `spatSample()` devuelve puntos **aleatorios**
dentro del polígono, no centros de celda. Dos estaciones cayeron en la misma celda,
así que son **68 celdas distintas, no 70**.


![Mapa predicción](resultados/fig14_mapa_prediccion.png)

**Precipitación estimada.** Rango 34,6 - 223,2 mm contra 34,9 - 224,8 observados.
Media 88,9 contra 90,1 — sesgo de −1,4 %.


![Observado vs estimado](resultados/fig17_observado_vs_estimado.png)

**La prueba visual.** (A) CHIRPS real. (B) reconstruido desde solo 70 puntos. Mismo
gradiente, mismas magnitudes, algo más suave — que es lo que se espera del kriging.


![Mapa incertidumbre](resultados/fig15_mapa_incertidumbre.png)

**Incertidumbre.** Se comporta como manda la teoría: **mínima en las estaciones**
(1,1 mm), **crece con la distancia**, **máxima en los bordes** y en los huecos sin
muestrear (12,6 mm) — filo oriental, punta norte, la "cintura" del departamento.

Este mapa solo es interpretable porque `sd_z = 1,06`. Con el modelo de menor RMSE
(`sd_z = 0,69`) habría sobreestimado la incertidumbre en un 45 %.


![Descomposición](resultados/fig16_descomposicion.png)

**La respuesta gráfica al enunciado.** (A) lo que explica la tendencia
determinística; (B) lo que aporta el kriging del residuo. El panel B **no es ruido**:
tiene estructura espacial organizada, y es exactamente el "papel de la correlación
espacial" que el enunciado pide evaluar.


![Mapa error](resultados/fig18_mapa_error.png)

**Error de reconstrucción** (observado − estimado) sobre toda la grilla. Los errores
grandes se concentran donde la incertidumbre predicha también era alta — otra
señal de que el modelo sabe dónde no sabe.


---
# Auditoría (12-sep-2026)

## Pruebas numéricas del kriging — 6 de 6

```
1. Interpola exacto en los datos    dif 3.55e-15,  var = 0     OK
2. Pesos suman 1                    1.000000000000            OK
3. Varianza nunca negativa          sd mín 1.14 mm            OK
4. Sigma definida positiva          autovalor mín +9.13       OK
5. Meseta / varianza                1.018                     OK
6. Sesgo en la media               -1.24 mm (-1.4 %)          OK
```

La prueba 1 es la fuerte: con pepita 0 el kriging **debe** interpolar exacto.

## Reproducibilidad

Clon limpio en otro directorio: **26 figuras byte-idénticas**, resultados iguales.
El script descomprime el dataset solo si hace falta.

## Whitelist

Se comparó **cada función** contra base R y los scripts del profesor. Sobrevivían
dos sin precedente, ambas reemplazadas:

| | Reemplazo | Verificación |
|---|---|---|
| `expanse()` | conteo de celdas × área de celda | 21 119 vs 21 125 km² (0,03 %) |
| `cellFromXY()` | aritmética columna/fila | resultado idéntico |

**El script usa `terra` + base R puro.**


---
# Errores encontrados (y resueltos)

Cuatro de los seis fallan **en silencio**.

| # | Error | ¿Avisa? |
|---|---|---|
| 1 | `ifelse()` en la estandarización de `W` (script del profesor) | no |
| 2 | `as.vector(ext())` trae nombres → `legend(NA, NA)` no dibuja | no |
| 3 | `terra::plot` recorta las leyendas de `"topleft"` | no |
| 4 | Denominador equivocado al medir cobertura | no, pero invierte la conclusión |
| 5 | Modelo gaussiano → matriz casi singular | no, devuelve números absurdos |
| 6 | `spatSample()` da puntos, no centros de celda | no, el conteo salía mal |

### 1. Bug en `Script_Spatial_Analytics.R` línea 77

```r
W <- matrix(ifelse(row_sums == 0, 0, W / row_sums), nrow = n, ncol = n)
```
`ifelse()` devuelve el largo del test (`n`), no del resultado (`n²`). Recicla la
primera columna. **Solución:** `sweep()`.

### 2. `as.vector(ext())` trae nombres

```r
e  <- as.vector(ext(borde))     # nombres: xmin xmax ymin ymax
lg <- c(x = e[1], y = e[4])     # el nombre queda "x.xmin", NO "x"
lg["x"]                         # -> NA
legend(NA, NA, ...)             # no dibuja nada y NO da error
```
**Solución:** `unname()`.

### 4. El denominador (Paso 1.3)

Medir cobertura contra el rectángulo en vez del departamento descartaba la mejor
covariable. **Define la máscara antes de calcular porcentajes.**

### 5. El gaussiano

Autovalor mínimo `4.8e-05` contra `16.9` del esférico. RMSE de 21 a **160 mm**,
R² = **−8,9**. **Solución:** elegir el modelo por validación, no por ajuste.

### 6. `spatSample()`

Devuelve puntos aleatorios dentro del polígono. Comparar coordenadas con centros de
celda nunca acierta. **Solución:** índice de celda desde la esquina y la resolución.


---
# Estado y pendientes

| Paso | Qué | Bloque | Estado |
|---|---|---|---|
| 1 | Auditoría de datos | — | ✅ |
| 2 | Borde y muestreo | — | ✅ |
| 3 | EDA espacial | **1. EDA** | ✅ |
| 4 | Tendencia y confusión | **2. Modelación** | ✅ |
| 5 | Moran, Geary y correlograma | **1. EDA** | ✅ |
| 6 | Semivariograma: 2 tendencias × 3 modelos | **2. Modelación** | ✅ |
| 7 | Kriging y validación cruzada | **3. Validación** | ✅ |
| 8 | Mapas de predicción e incertidumbre | **4. Predicción** | ✅ |
| 9 | **Documento PDF ≤ 10 páginas** | — | ✅ **9 páginas** |

## El documento

`Proyecto 1 AnalíticaDeDatos.pdf`, en la raíz del repo. Se regenera así:

```bash
Rscript R/proyecto1.R      # solo si cambiaron las figuras
informe/generar_pdf.sh
```

La fuente es `informe/informe.html`: HTML con reglas `@page` de CSS, impreso por
el motor del navegador. **No es R Markdown**, que el enunciado prohíbe. Una sola
columna, A4, 9 de las 10 páginas permitidas.

Lleva 9 figuras y 9 tablas, todas interpretadas en el texto —el enunciado avisa
que presentar gráficos sin interpretación no cuenta—. Cierra con limitaciones y
referencias.

## Repaso 3D para la sustentación

Fuera del repo, en `~/Projects/Proyecto1_Repaso3D`, quedó un visor que recorre
los diez estados del análisis en 3D: relieve, lluvia observada, estaciones,
tendencia, residuales, autocorrelación, semivariograma, predicción,
incertidumbre y error.

```bash
cd ~/Projects/Proyecto1_Repaso3D && ./servir.sh
```

No se metió en este repo a propósito: usa `jsonlite` y `three.js`, y el `.R` que
se entrega tiene que quedarse en `terra` + base R.

### Pendientes

- 🔴 En `main`, `R/proyecto1.R` fue **reemplazado** por el script antiguo
  (587 líneas, declara `sp`, `gstat`, `corrplot`). El análisis de este cuaderno
  vive en la rama `feat/analisis-cesar`. **Hay que resolverlo con el grupo.**
- Correos institucionales vacíos en la tabla de integrantes.
- `prompts_IA.docx` si se declara uso de IA.

### Recordatorio de entrega

| | |
|---|---|
| **Plazo** | domingo **13-sep-2026, 23:59** — solo Moodle |
| **Documento** | PDF, máx. **10 páginas**, una sola columna |
| **Código** | **un único** `.R`, reproducible, en archivo aparte |
| **Prohibido** | R Markdown, solo-Colab, manuscrito |
| **Sustentación** | se sortea un integrante; ausencia sin excusa = **0.0** |